<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW15 · Session 2 · PCA, t-SNE, and Unsupervised Validation

        **Course level:** developing beginner  
        **Prior learning:** scaled customer features and clustering from TW15 Session 1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Dimensionality reduction creates lower-dimensional representations. PCA supports variance-based compression and interpretation; t-SNE supports local visual exploration. Neither automatically proves true clusters.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - fit and interpret principal component analysis
- use explained variance and loadings
- create a reproducible t-SNE visualisation
- distinguish visual separation from cluster validation
- assess silhouette, stability, algorithm agreement, and external reference agreement


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load and standardise customer features

Reference labels remain outside the unsupervised feature matrix.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

customers = pd.read_csv(DATA_DIR / "customer_segments.csv")
feature_columns = [
    "annual_income", "annual_spend",
    "visits_per_month", "digital_engagement",
]
X = customers[feature_columns]
X_scaled = StandardScaler().fit_transform(X)
reference = customers["reference_segment"]
print(X.shape)


## 1. Principal Component Analysis

PCA creates orthogonal linear combinations that capture decreasing amounts of variance. It is sensitive to scale, so standardisation is usually required when features use different units.


### Fit all principal components

With four source features, PCA can create at most four components.


In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained = pd.DataFrame({
    "component": [f"PC{index}" for index in range(1, 5)],
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca_full.explained_variance_ratio_),
})
print(explained.round(3))


### Plot explained and cumulative variance

The chart supports a transparent component-count decision.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(explained["component"], explained["explained_variance_ratio"], label="Individual")
ax.plot(
    explained["component"],
    explained["cumulative_variance"],
    marker="o",
    color="#dc2626",
    label="Cumulative",
)
ax.set_ylim(0, 1.05)
ax.set_title("PCA explained variance")
ax.set_ylabel("Proportion of variance")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)


### Inspect component loadings

Loadings show how strongly each standardised feature contributes to a component. Component signs may flip without changing the solution.


In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=feature_columns,
    columns=[f"PC{index}" for index in range(1, 5)],
)
print(loadings.round(3))


### Create a two-component representation

This projection is useful for visualisation but does not preserve all information.


In [ ]:
pca_two = PCA(n_components=2)
X_pca_two = pca_two.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_pca_two[:, 0], X_pca_two[:, 1], alpha=0.65)
ax.set_title("Customers in two principal components")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.tight_layout()
plt.show()
plt.close(fig)


## 2. t-SNE for local visual exploration

t-SNE preserves local neighbourhoods for a low-dimensional visualisation. Global distances, cluster sizes, and gaps can be distorted. Results depend on perplexity, initialisation, and random seed, so t-SNE is not itself a clustering algorithm or proof of cluster count.


### Create a reproducible t-SNE embedding

Perplexity must be smaller than the number of observations. random_state fixes this teaching run.


In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    max_iter=750,
    random_state=42,
)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.65)
ax.set_title("t-SNE visualisation")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. Validate beyond a picture

Useful checks include:

- internal separation such as silhouette;
- stability across random starts or resamples;
- agreement across reasonable algorithms;
- interpretability in original units;
- external reference agreement when a legitimate reference exists;
- business usefulness and fairness review.


### Fit a reference k-means solution

The clustering still uses original scaled features, not the t-SNE coordinates.


In [ ]:
main_kmeans = KMeans(n_clusters=3, n_init=30, random_state=42)
main_labels = main_kmeans.fit_predict(X_scaled)
main_silhouette = silhouette_score(X_scaled, main_labels)

print("Silhouette:", round(main_silhouette, 3))


### Check stability across initialisation seeds

Adjusted Rand Index compares each alternative assignment with the main assignment.


In [ ]:
stability_rows = []
for seed in [1, 7, 21, 99, 2026]:
    alternative = KMeans(n_clusters=3, n_init=1, random_state=seed)
    alternative_labels = alternative.fit_predict(X_scaled)
    stability_rows.append({
        "seed": seed,
        "ARI_vs_main": adjusted_rand_score(main_labels, alternative_labels),
        "silhouette": silhouette_score(X_scaled, alternative_labels),
    })

stability_table = pd.DataFrame(stability_rows)
print(stability_table.round(3))


### Use the held-aside reference only for evaluation

Adjusted Rand Index evaluates agreement without making the reference part of fitting. A real external label must itself be credible.


In [ ]:
external_ari = adjusted_rand_score(reference, main_labels)
print("ARI against reference segment:", round(external_ari, 3))


### Overlay cluster labels on both projections

Colour supports inspection, but the validation metrics come from the original scaled space.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X_pca_two[:, 0], X_pca_two[:, 1], c=main_labels, cmap="viridis", alpha=0.7)
axes[0].set_title("PCA projection")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=main_labels, cmap="viridis", alpha=0.7)
axes[1].set_title("t-SNE visualisation")
axes[1].set_xlabel("Dimension 1")
axes[1].set_ylabel("Dimension 2")

fig.suptitle("Same cluster labels, different projections")
fig.tight_layout()
plt.show()
plt.close(fig)


## Guided validation report

A strong report states the feature space, scaling, algorithm and parameters, candidate k values, internal scores, stability evidence, profiles, external checks, limitations, and intended use.


### Assemble evidence without overclaiming

The report remains provisional even when metrics are strong.


In [ ]:
validation_report = pd.Series({
    "observations": len(customers),
    "features": len(feature_columns),
    "chosen_k": 3,
    "silhouette": main_silhouette,
    "minimum_seed_ARI": stability_table["ARI_vs_main"].min(),
    "external_reference_ARI": external_ari,
    "intended_use": "Teaching exploration only",
    "major_limitation": "Synthetic data and algorithm-dependent segments",
})
print(validation_report)


## Student coding lab

Fit unsupervised methods only on the four feature columns; use the reference label after fitting.


        ### Student task 1: Analyse PCA variance

        <div class="task"><strong>Your job:</strong> Fit full PCA, create explained and cumulative variance tables, and choose the smallest component count reaching at least 90%.</div>

        **Check your work**

        - All four component ratios are reported.
- Cumulative variance is calculated.
- The selection rule is programmatic.


In [ ]:
# STUDENT TASK 1
task_pca = PCA()
task_scores = task_pca.fit_transform(X_scaled)
task_explained = pd.DataFrame()
chosen_components = 0
print(task_explained)
print("Chosen components:", chosen_components)


        ### Student task 2: Interpret loadings

        <div class="task"><strong>Your job:</strong> Create a loading table and identify the two largest absolute contributors to PC1 and PC2.</div>

        **Check your work**

        - Rows use original feature names.
- Absolute magnitude determines contribution.
- Interpretation recognises arbitrary sign.


In [ ]:
# STUDENT TASK 2
task_loadings = pd.DataFrame()
top_contributors = {}
print(task_loadings)
print(top_contributors)


        ### Student task 3: Compare t-SNE settings

        <div class="task"><strong>Your job:</strong> Create embeddings with perplexity 10 and 40 using the same seed. Plot side by side and state what can and cannot be concluded.</div>

        **Check your work**

        - Both embeddings use the same scaled input.
- Perplexities are valid for sample size.
- The comment avoids treating visual gaps as proof.


In [ ]:
# STUDENT TASK 3
embedding_10 = np.empty((0, 2))
embedding_40 = np.empty((0, 2))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Fit and plot both embeddings.
plt.close(fig)


        ### Student task 4: Measure cluster stability

        <div class="task"><strong>Your job:</strong> Fit k-means with five different seeds and n_init=1. Report pairwise or main-solution ARI plus silhouette.</div>

        **Check your work**

        - All solutions use k=3.
- ARI ignores arbitrary label numbering.
- Both stability and separation are interpreted.


In [ ]:
# STUDENT TASK 4
task_stability = []
print(pd.DataFrame(task_stability))


        ### Student task 5: Write an unsupervised validation report

        <div class="task"><strong>Your job:</strong> Combine k choice, silhouette, stability, PCA evidence, external-reference ARI, profile interpretation, intended use, and limitations.</div>

        **Check your work**

        - Every field is supported by computed evidence.
- The reference was excluded from fitting.
- The report avoids calling clusters objective truths.



**Optional extension:** Use bootstrap resamples and match cluster centroids before comparing assignments.


In [ ]:
# STUDENT TASK 5
unsupervised_report = {
    "chosen_k": None,
    "silhouette": None,
    "stability": None,
    "pca_evidence": None,
    "external_evidence": None,
    "intended_use": "",
    "limitations": [],
}
print(unsupervised_report)


        ## Knowledge check

        Answer these without running new code first.

        1. What does PCA optimise?
2. What do PCA loadings represent?
3. Why should t-SNE global distances be treated cautiously?
4. What does adjusted Rand index compare?
5. Why is a visually separated plot insufficient validation?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW16 introduces datetime indexes, lags, rolling measures, growth rates, and changes of time-series frequency.
